# Learning RAG System - Flashcard Feature

## 1. Environment Setup

Install dependencies and download the required data files.

In [1]:
!pip uninstall -y torchcodec
!pip install -q langchain langchain-community langchain-core langchain-google-genai langchain-huggingface langchain-openai langchain-qdrant langchain-text-splitters loguru pypdf qdrant-client ragas sentence-transformers transformers typer vllm

!mkdir -p data
!gdown 1PqUYjfhdR8xBqj4mB7bW5X-yreYWqlZY -O data/data.zip
!unzip -o data/data.zip -d data

Found existing installation: torchcodec 0.10.0+cu128
Uninstalling torchcodec-0.10.0+cu128:
  Successfully uninstalled torchcodec-0.10.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.4/244.4 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 85.3 MB

## 2. Core Imports

Load the standard libraries and validation utilities used across the notebook.

In [2]:
from pathlib import Path
from loguru import logger
import json
import re

from pydantic import BaseModel, Field
from typing import Literal, List, Optional
from pydantic_settings import BaseSettings

## 3. Data Schemas

Define typed objects for retrieved chunks, citations, and flashcards.

In [3]:
class ChunkMetadata(BaseModel):
    document_id: str
    filename: str
    source: str
    page: int
    chunk_id: str
    section: Optional[str] = None

class RetrievedChunk(BaseModel):
    text: str
    score: float
    metadata: ChunkMetadata

class Citation(BaseModel):
    source_index: int
    source_marker: str
    filename: str
    page: int
    section: Optional[str] = None
    chunk_id: str

class Flashcard(BaseModel):
    front: str
    back: str
    hint: Optional[str] = None
    topic: Optional[str] = None
    source_markers: List[str]

class FlashcardSet(BaseModel):
    scope: str
    target: Optional[str] = None
    cards: List[Flashcard]
    citations: List[Citation]

## 4. System Configuration

Set paths, embedding model, LLM provider, and generation defaults.

In [4]:
class Settings(BaseSettings):
    data_dir: Path = Path("data")
    storage_dir: Path = Path("storage/qdrant")
    qdrant_collection: str = "rag_chunks"

    chunk_size: int = Field(default=1000, ge=100)
    chunk_overlap: int = Field(default=150, ge=0)
    top_k: int = Field(default=5, ge=1, le=64)

    embedding_model: str = "keepitreal/vietnamese-sbert"

    llm_provider: Literal["hf_local", "gemini", "vllm"] = "hf_local"
    llm_temperature: float = Field(default=0.1, ge=0.0, le=2.0)

    hf_model: str = "Qwen/Qwen2.5-1.5B-Instruct"
    hf_device: int = 0
    hf_max_new_tokens: int = Field(default=1024, ge=1)

    gemini_model: str = "gemini-2.5-flash"

    generation_retrieval_k: int = Field(default=16, ge=1, le=128)
    flashcards_default_count: int = Field(default=5, ge=1, le=100)

settings = Settings()
settings.data_dir.mkdir(parents=True, exist_ok=True)
settings.storage_dir.mkdir(parents=True, exist_ok=True)

## 5. RAG Dependencies

Import libraries for PDF loading, chunking, embeddings, and Qdrant storage.

In [5]:
import hashlib
import uuid
from collections import defaultdict

from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http import models as qmodels

## 6. Embedding Model

Create the embedding model used to convert text chunks into vectors.

In [6]:
def get_embeddings():
    return HuggingFaceEmbeddings(
        model_name=settings.embedding_model,
        model_kwargs={"device": "cuda"},
        encode_kwargs={"normalize_embeddings": True},
    )

## 7. Vector Store Setup

Create or reuse the local Qdrant collection for storing document vectors.

In [7]:
def get_client() -> QdrantClient:
    return QdrantClient(path=str(settings.storage_dir))

def ensure_collection(recreate: bool = False):
    client = get_client()
    if recreate and client.collection_exists(settings.qdrant_collection):
        client.delete_collection(settings.qdrant_collection)

    if not client.collection_exists(settings.qdrant_collection):
        dim = len(get_embeddings().embed_query("test"))
        client.create_collection(
            collection_name=settings.qdrant_collection,
            vectors_config=qmodels.VectorParams(
                size=dim, distance=qmodels.Distance.COSINE),
        )

def get_vector_store():
    return QdrantVectorStore(
        client=get_client(),
        collection_name=settings.qdrant_collection,
        embedding=get_embeddings(),
    )

## 8. PDF Chunk Builder

Load PDF pages, attach metadata, and split text into retrievable chunks.

In [8]:
def build_chunks(pdf_paths):
    page_docs = []

    for path in pdf_paths:
        loader = PyPDFLoader(str(path))
        pages = loader.load()

        doc_id = hashlib.sha1(f"{path.name}:{path.stat().st_size}".encode("utf-8")).hexdigest()[:16]

        for doc in pages:
            doc.metadata = {
                "document_id": doc_id,
                "filename": path.name,
                "source": str(path.resolve()),
                "page": int(doc.metadata.get("page", 0)) + 1,
            }
        page_docs.extend(pages)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=settings.chunk_size,
        chunk_overlap=settings.chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(page_docs)

    per_doc_counter = defaultdict(int)
    for chunk in chunks:
        doc_id = chunk.metadata["document_id"]
        idx = per_doc_counter[doc_id]
        per_doc_counter[doc_id] += 1
        chunk.metadata["chunk_id"] = f"{doc_id}:{chunk.metadata['page']}:{idx}"

    return chunks

## 9. Single PDF Ingestion

Ingest one PDF file into the vector store.

In [9]:
def save_and_ingest_pdf(file_path: str):
    path = Path(file_path)
    ensure_collection(recreate=False)
    chunks = build_chunks([path])
    if not chunks:
        return 0

    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, c.metadata["chunk_id"]))
        for c in chunks]
    get_vector_store().add_documents(chunks, ids=ids)
    logger.info(f"Saved {len(chunks)} chunks from {path.name}")
    return len(chunks)

## 10. Batch PDF Ingestion

Find all PDFs in the data folder and ingest them into Qdrant.

In [10]:
def ingest_data_directory():
    ensure_collection(recreate=False)
    pdf_paths = list(settings.data_dir.glob("*.pdf"))
    if not pdf_paths:
        logger.warning(f"No PDF files found in directory: {settings.data_dir}")
        return 0

    logger.info(f"Found {len(pdf_paths)} PDF files. Proceeding with chunking...")
    chunks = build_chunks(pdf_paths)
    if not chunks:
        return 0

    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, c.metadata["chunk_id"]))
        for c in chunks]
    get_vector_store().add_documents(chunks, ids=ids)

    logger.info(f"Ingested {len(chunks)} chunks from {len(pdf_paths)} documents.")
    return len(chunks)

## 11. LLM Dependencies

Import model wrappers for local Hugging Face and Gemini backends.

In [11]:
import torch
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

## 12. Local Hugging Face LLM

Build a local text-generation model for offline inference.

In [12]:
def _build_hf_local():
    logger.info(f"Loading local model: {settings.hf_model}...")
    tokenizer = AutoTokenizer.from_pretrained(settings.hf_model)
    model = AutoModelForCausalLM.from_pretrained(
        settings.hf_model,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

    text_gen_pipeline = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=settings.hf_max_new_tokens,
        do_sample=settings.llm_temperature > 0,
        temperature=settings.llm_temperature,
        return_full_text=False,
    )

    llm = HuggingFacePipeline(pipeline=text_gen_pipeline)
    return ChatHuggingFace(llm=llm)

## 13. Gemini LLM

Build a Gemini chat model as an alternative generation backend.

In [13]:
def _build_gemini():
    return ChatGoogleGenerativeAI(
        model=settings.gemini_model,
        temperature=settings.llm_temperature,
    )

## 14. LLM Provider Selector

Select the configured model backend at runtime.

In [14]:
def get_llm():
    provider = settings.llm_provider
    if provider == "hf_local":
        return _build_hf_local()
    elif provider == "gemini":
        return _build_gemini()
    else:
        raise ValueError(f"Provider {provider} is not supported.")

## 15. Retrieval and Citations

Retrieve relevant chunks and convert them into citation metadata.

In [15]:
def retrieve(query: str, k: int = 5):
    store = get_vector_store()
    hits = store.similarity_search_with_score(query=query, k=k)
    return [
        RetrievedChunk(
            text=doc.page_content,
            score=float(score),
            metadata=ChunkMetadata(**doc.metadata),
        )
        for doc, score in hits
    ]

def format_citations(chunks):
    return [
        Citation(
            source_index=i,
            source_marker=f"S{i}",
            filename=c.metadata.filename,
            page=c.metadata.page,
            chunk_id=c.metadata.chunk_id,
        )
        for i, c in enumerate(chunks, start=1)
    ]

## 16. JSON Output Parser

Extract structured JSON from LLM responses.

In [16]:
def _parse_json(text: str) -> dict:
    match = re.search(r'\{.*\}', text, re.DOTALL)

    if match:
        json_str = match.group(0)
        try:
            return json.loads(json_str)
        except json.JSONDecodeError as e:
            logger.error(f"Internal JSON parsing error: {e}")
            logger.error(f"Faulty string: {json_str}")
            return {}
    else:
        logger.error(f"No JSON structure found in LLM output: {text}")
        return {}


## 17. Flashcard Prompt Template

Build a grounded prompt for study flashcard generation.

In [17]:
def build_flashcards_prompt(chunks: list, count: int) -> str:
    context_text = "\n".join([
        f"---\n[source=S{i+1}] ({chunk.metadata.filename} p.{chunk.metadata.page})\n{chunk.text}"
        for i, chunk in enumerate(chunks)
    ])
    prompt = f"""You are generating study flashcards grounded in the provided context.

Rules:
- Generate exactly {count} flashcards.
- Fronts should ask for a concept, term, distinction, formula, process, or short explanation.
- Backs must be concise but complete enough for revision.
- Ground every card in the context. Do not invent facts.
- Avoid low-value cards that only restate section headings.
- Avoid duplicate or overly similar cards.
- Provide at least one [S#] source marker per card when possible.
- If the context is insufficient for {count} useful cards, produce fewer rather than pad.
- Write all fronts, backs, hints, and topic labels in English.

Output STRICTLY valid JSON and no other text, with this exact shape:
{{
  "cards": [
    {{
      "front": "string",
      "back": "string",
      "hint": "optional string or null",
      "topic": "optional short topic label or null",
      "source_markers": ["S1"]
    }}
  ]
}}

Context:
{context_text}

Respond with ONLY the JSON object."""
    return prompt

In [30]:
# question = "Phương pháp LoRA là gì?"
# chunks = retrieve(question, k=2)
# flash_prompt = build_flashcards_prompt(chunks, 3)
# print(flash_prompt)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

You are generating study flashcards grounded in the provided context.

Rules:
- Generate exactly 3 flashcards.
- Fronts should ask for a concept, term, distinction, formula, process, or short explanation.
- Backs must be concise but complete enough for revision.
- Ground every card in the context. Do not invent facts.
- Avoid low-value cards that only restate section headings.
- Avoid duplicate or overly similar cards.
- Provide at least one [S#] source marker per card when possible.
- If the context is insufficient for 3 useful cards, produce fewer rather than pad.
- Write all fronts, backs, hints, and topic labels in English.

Output STRICTLY valid JSON and no other text, with this exact shape:
{
  "cards": [
    {
      "front": "string",
      "back": "string",
      "hint": "optional string or null",
      "topic": "optional short topic label or null",
      "source_markers": ["S1"]
    }
  ]
}

Context:
---
[source=S1] ([Description]-LLMs-Fine-tuning.pdf p.6)
SFT thường ảnh hưởng

## 18. Flashcard Generation Feature

Generate study flashcards from retrieved document context.

In [26]:
def generate_flashcards(query: str, count: int = 3):
    chunks = retrieve(query, k=settings.generation_retrieval_k)
    prompt = build_flashcards_prompt(chunks, count)

    response = get_llm().invoke([HumanMessage(content=prompt)])
    payload = _parse_json(response.content)

    cards = [Flashcard(**card) for card in payload.get("cards", [])]
    return FlashcardSet(scope="query",
    target=query, cards=cards, citations=format_citations(chunks))

In [31]:
question = "Phương pháp LoRA là gì?"
flash_res = generate_flashcards(question, 1)

print(flash_res.cards)
print(f"\n Front of card: {flash_res.cards[0].front}")
print(f"\n Back of card: {flash_res.cards[0].back}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-05-08 02:05:41.799 | INFO     | __main__:_build_hf_local:2 - Loading local model: Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Flashcard(front='fine-tuning process', back='fine-tuning involves supervised fine-tuning, LoRA, and QLoRA to optimize performance.', hint=None, topic=None, source_markers=['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11'])]

 Front of card: fine-tuning process

 Back of card: fine-tuning involves supervised fine-tuning, LoRA, and QLoRA to optimize performance.


## 19. End-to-End Demo

Run ingestion and test the flashcard learning feature.

In [29]:
import urllib.request
import shutil

if settings.storage_dir.exists():
    try:
        client = get_client()
        client.close()
    except Exception:
        logger.warning("Storage is locked. Attempting to clear storage directory for a clean start.")
        shutil.rmtree(settings.storage_dir, ignore_errors=True)
        settings.storage_dir.mkdir(parents=True, exist_ok=True)

settings.data_dir.mkdir(parents=True, exist_ok=True)
logger.info("STEP 1: CHECKING & PREPARING DATA AT /content/data")

if not any(settings.data_dir.iterdir()):
    logger.info("Directory is empty, downloading sample PDF for testing...")
    sample_url = "https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf"
    urllib.request.urlretrieve(sample_url, settings.data_dir / "sample_dummy.pdf")
else:
    logger.info(f"Found documents in {settings.data_dir}. Skipping sample download.")

logger.info("STEP 2: INGESTING DIRECTORY INTO VECTOR STORE (QDRANT)")
total_chunks = ingest_data_directory()
print(f"Successfully chunked and saved {total_chunks} text segments into Qdrant.\n")

if total_chunks > 0:
    logger.info("STEP 3: RUNNING FLASHCARD FEATURE")

    print("\n" + "=" * 55)
    print("FEATURE: GENERATE FLASHCARDS (FLASHCARDS)")
    print("=" * 55)
    topic = "Phương pháp LoRA là gì?"
    flash_res = generate_flashcards(topic, count=1)

    for idx, card in enumerate(flash_res.cards, 1):
        print(f"Card {idx}:")
        print(f"  Front (Question): {card.front}")
        print(f"  Back (Answer)   : {card.back}")
        if card.hint:
            print(f"  Hint            : {card.hint}")
        print("-" * 30)

    logger.info("FLASHCARD GENERATION COMPLETED SUCCESSFULLY!")

2026-05-08 02:04:14.583 | INFO     | __main__:<cell line: 0>:14 - STEP 1: CHECKING & PREPARING DATA AT /content/data
2026-05-08 02:04:14.586 | INFO     | __main__:<cell line: 0>:21 - Found documents in data. Skipping sample download.
2026-05-08 02:04:14.589 | INFO     | __main__:<cell line: 0>:23 - STEP 2: INGESTING DIRECTORY INTO VECTOR STORE (QDRANT)
2026-05-08 02:04:14.681 | INFO     | __main__:ingest_data_directory:8 - Found 7 PDF files. Proceeding with chunking...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-05-08 02:04:50.550 | INFO     | __main__:ingest_data_directory:17 - Ingested 340 chunks from 7 documents.
2026-05-08 02:04:50.552 | INFO     | __main__:<cell line: 0>:28 - STEP 3: RUNNING FLASHCARD FEATURE


Successfully chunked and saved 340 text segments into Qdrant.


FEATURE: GENERATE FLASHCARDS (FLASHCARDS)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-05-08 02:04:53.291 | INFO     | __main__:_build_hf_local:2 - Loading local model: Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
2026-05-08 02:05:12.356 | INFO     | __main__:<cell line: 0>:44 - FLASHCARD GENERATION COMPLETED SUCCESSFULLY!


Card 1:
  Front (Question): fine-tuning process
  Back (Answer)   : fine-tuning involves supervised fine-tuning, LoRA, and QLoRA to optimize performance.
------------------------------
